# CodonFM SAE — Dashboard Export

The interactive dashboard provides a visual interface for exploring SAE features. It shows:

- A **UMAP scatter plot** of all features, computed from the SAE decoder weight vectors. Features that decode to similar directions in activation space appear nearby, revealing natural clusters (e.g., codon-usage features vs. domain-specific features).
- **Feature cards** with top-activating sequences, where each codon is colored by activation intensity. This lets you see *what* a feature responds to in context.
- **Crossfiltering** by metadata columns — GSEA labels, codon annotations, gene families, variant analysis scores.

This notebook generates the three data files the dashboard needs:

| File | Contents |
|---|---|
| `features_atlas.parquet` | One row per feature: UMAP coordinates, activation frequency, max activation, cluster ID, plus any enrichment columns |
| `feature_metadata.parquet` | Per-feature description and stats (activation frequency, max activation) |
| `feature_examples.parquet` | Top-K activating sequences per feature, with per-codon activation arrays |

## Setup

In [ ]:
# Config
SAE_CHECKPOINT = "../outputs/1b_layer16/checkpoints/checkpoint_final.pt"
MODEL_PATH = "../../../../../../../checkpoints/NV-CodonFM-Encodon-TE-Cdwt-1B-v1"
CSV_PATH = "../../../../../../../codonfm/data/codonfm_ref_only.csv"
LAYER = 16
CONTEXT_LENGTH = 2048
BATCH_SIZE = 8
DEVICE = "cuda"
NUM_SEQUENCES = 2000
N_EXAMPLES = 6  # Top examples per feature

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import torch


_REPO_ROOT = Path("..").resolve().parent.parent.parent.parent.parent
_CODONFM_TE_DIR = _REPO_ROOT / "recipes" / "codonfm_ptl_te"
sys.path.insert(0, str(_CODONFM_TE_DIR))
sys.path.insert(0, str(Path("..").resolve()))

from codonfm_sae.data import read_codon_csv
from sae.architectures import TopKSAE
from sae.utils import set_seed
from src.data.preprocess.codon_sequence import process_item
from src.inference.encodon import EncodonInference


set_seed(42)
device = DEVICE if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Load SAE, Model, and Data

In [ ]:
ckpt = torch.load(SAE_CHECKPOINT, map_location="cpu", weights_only=False)
state_dict = ckpt["model_state_dict"]
if any(k.startswith("module.") for k in state_dict):
    state_dict = {k.removeprefix("module."): v for k, v in state_dict.items()}

input_dim = ckpt.get("input_dim") or state_dict["encoder.weight"].shape[1]
hidden_dim = ckpt.get("hidden_dim") or state_dict["encoder.weight"].shape[0]
model_config = ckpt.get("model_config", {})
top_k = model_config.get("top_k")

sae = TopKSAE(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    top_k=top_k,
    normalize_input=model_config.get("normalize_input", False),
)
sae.load_state_dict(state_dict)
sae = sae.eval().to(device)

print(f"SAE: {input_dim} -> {hidden_dim:,} features (top-{top_k})")

In [ ]:
inference = EncodonInference(
    model_path=MODEL_PATH,
    task_type="embedding_prediction",
    use_transformer_engine=True,
)
inference.configure_model()
inference.model.to(device).eval()

num_layers = len(inference.model.model.layers)
target_layer = LAYER if LAYER >= 0 else num_layers + LAYER
print(f"Encodon: {num_layers} layers, target layer {target_layer}")

In [ ]:
records = read_codon_csv(CSV_PATH, max_sequences=NUM_SEQUENCES, max_codons=CONTEXT_LENGTH - 2)
sequences = [r.sequence for r in records]
sequence_ids = [r.id for r in records]
print(f"Loaded {len(sequences)} sequences")

## Extract Activations

We extract 3D activations (sequences x positions x hidden_dim) from the target layer, stripping the CLS and SEP tokens. These are the raw inputs the SAE was trained on.

In [ ]:
from tqdm import tqdm


all_embeddings = []
all_masks = []

n_batches = (len(sequences) + BATCH_SIZE - 1) // BATCH_SIZE

with torch.no_grad():
    for i in tqdm(range(0, len(sequences), BATCH_SIZE), total=n_batches, desc="Extracting activations"):
        batch_seqs = sequences[i : i + BATCH_SIZE]
        items = [process_item(s, context_length=CONTEXT_LENGTH, tokenizer=inference.tokenizer) for s in batch_seqs]

        batch = {
            "input_ids": torch.tensor(np.stack([it["input_ids"] for it in items])).to(device),
            "attention_mask": torch.tensor(np.stack([it["attention_mask"] for it in items])).to(device),
        }

        out = inference.model(batch, return_hidden_states=True)
        hidden = out.all_hidden_states[LAYER].float().cpu()
        attn_mask = batch["attention_mask"].cpu()

        # Build mask excluding CLS (pos 0) and SEP (last real pos)
        keep = attn_mask.clone()
        keep[:, 0] = 0
        lengths = attn_mask.sum(dim=1)
        for b in range(keep.shape[0]):
            sep = int(lengths[b].item()) - 1
            if sep > 0:
                keep[b, sep] = 0

        all_embeddings.append(hidden)
        all_masks.append(keep)

        del out, batch
        torch.cuda.empty_cache()

# Pad to same length
max_len = max(e.shape[1] for e in all_embeddings)
padded_emb = []
padded_masks = []
for emb, msk in zip(all_embeddings, all_masks):
    B, L, D = emb.shape
    if L < max_len:
        emb = torch.cat([emb, torch.zeros(B, max_len - L, D)], dim=1)
        msk = torch.cat([msk, torch.zeros(B, max_len - L, dtype=msk.dtype)], dim=1)
    padded_emb.append(emb)
    padded_masks.append(msk)

activations = torch.cat(padded_emb, dim=0)
masks = torch.cat(padded_masks, dim=0)
activations_flat = activations[masks.bool()]

print(f"Activations: {activations.shape}")
print(f"Valid codons: {activations_flat.shape[0]:,}")

## Compute Feature Statistics

For each feature, compute global statistics (activation frequency, mean activation, max activation) from the flattened activations. These become columns in the atlas and are used for filtering in the dashboard.

In [ ]:
from sae.analysis import compute_feature_stats, compute_feature_umap, save_feature_atlas


stats, _ = compute_feature_stats(sae, activations_flat, device=device)
print(f"Computed stats for {len(stats)} features")
print(f"  Alive features: {sum(1 for s in stats.values() if s.get('activation_freq', 0) > 0)}")

## Compute UMAP from Decoder Weights

The UMAP is computed from the SAE **decoder weight matrix** (not from activations). Each feature has a decoder vector in activation space — the direction it represents. UMAP projects these high-dimensional vectors to 2D, preserving local structure.

Features with similar decoder vectors appear nearby on the UMAP, even if they fire on different sequences. This reveals the geometric structure the SAE has learned: clusters of synonymous-codon features, amino-acid features, domain-specific features, etc.

We also run HDBSCAN clustering on the UMAP coordinates to automatically identify feature groups.

In [ ]:
geometry = compute_feature_umap(
    sae,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
    compute_clusters=True,
    hdbscan_min_cluster_size=20,
)

n_clusters = len(set(geometry.get("cluster_id", {}).values())) - (
    1 if -1 in geometry.get("cluster_id", {}).values() else 0
)
print(f"UMAP computed: {len(geometry.get('umap_x', {}))} features")
print(f"HDBSCAN found {n_clusters} clusters")

## Export Feature Atlas

The atlas is the central parquet file — one row per feature with all metadata columns. The dashboard loads this to render the UMAP and populate filter controls.

In [ ]:
output_dir = Path("../outputs/1b_layer16/dashboard")
output_dir.mkdir(parents=True, exist_ok=True)

atlas_path = output_dir / "features_atlas.parquet"
save_feature_atlas(stats, geometry, atlas_path)
print(f"Saved atlas to {atlas_path}")

## Export Feature Examples

For each feature, we find the top-K sequences with the highest max activation and extract per-codon activation arrays. This is the data behind the "feature cards" in the dashboard — the highlighted sequences that show what each feature responds to.

The export uses a two-pass approach to avoid holding all per-codon activations in memory:
1. **Pass 1**: Compute max activation per (sequence, feature) to identify top examples.
2. **Pass 2**: Re-encode only the needed sequences to extract per-codon activations.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq


n_features = sae.hidden_dim
n_sequences = activations.shape[0]
valid_lens = masks.sum(dim=1).long()

# Pass 1: max activation per (sequence, feature)
print("Pass 1: Computing max activations per sequence...")
max_acts = torch.zeros(n_sequences, n_features)
for i in tqdm(range(n_sequences), desc="Max activations"):
    vl = int(valid_lens[i].item())
    if vl == 0:
        continue
    emb = activations[i, :vl, :].to(device)
    with torch.no_grad():
        _, codes = sae(emb)
    max_acts[i] = codes.max(dim=0).values.cpu()

# Find top examples per feature
print("Finding top examples per feature...")
top_indices = torch.topk(max_acts, k=min(N_EXAMPLES, n_sequences), dim=0).indices

# Build reverse index
needed_sequences = {}
for feat_idx in range(n_features):
    for rank in range(top_indices.shape[0]):
        seq_idx = int(top_indices[rank, feat_idx].item())
        if seq_idx not in needed_sequences:
            needed_sequences[seq_idx] = set()
        needed_sequences[seq_idx].add(feat_idx)

# Pass 2: extract per-codon activations for top examples
print(f"Pass 2: Extracting per-codon activations ({len(needed_sequences)} sequences)...")
example_acts = {}
for seq_idx in tqdm(sorted(needed_sequences.keys()), desc="Per-codon activations"):
    vl = int(valid_lens[seq_idx].item())
    if vl == 0:
        continue
    emb = activations[seq_idx, :vl, :].to(device)
    with torch.no_grad():
        _, codes = sae(emb)
    codes_cpu = codes.cpu()
    for feat_idx in needed_sequences[seq_idx]:
        example_acts[(seq_idx, feat_idx)] = codes_cpu[:, feat_idx].numpy().tolist()

# Write feature_metadata.parquet
print("Writing feature_metadata.parquet...")
meta_rows = []
for feat_idx in range(n_features):
    freq = (max_acts[:, feat_idx] > 0).float().mean().item()
    max_val = max_acts[:, feat_idx].max().item()
    meta_rows.append(
        {
            "feature_id": feat_idx,
            "description": f"Feature {feat_idx}",
            "activation_freq": freq,
            "max_activation": max_val,
        }
    )

meta_table = pa.table(
    {
        "feature_id": pa.array([r["feature_id"] for r in meta_rows], type=pa.int32()),
        "description": pa.array([r["description"] for r in meta_rows]),
        "activation_freq": pa.array([r["activation_freq"] for r in meta_rows], type=pa.float32()),
        "max_activation": pa.array([r["max_activation"] for r in meta_rows], type=pa.float32()),
    }
)
pq.write_table(meta_table, output_dir / "feature_metadata.parquet", compression="snappy")

# Write feature_examples.parquet
print("Writing feature_examples.parquet...")
example_rows = []
for feat_idx in range(n_features):
    for rank in range(top_indices.shape[0]):
        seq_idx = int(top_indices[rank, feat_idx].item())
        key = (seq_idx, feat_idx)
        if key not in example_acts:
            continue
        raw_seq = sequences[seq_idx]
        n_codons = len(raw_seq) // 3
        codon_seq = " ".join(raw_seq[j * 3 : (j + 1) * 3] for j in range(n_codons))
        example_rows.append(
            {
                "feature_id": feat_idx,
                "example_rank": rank,
                "protein_id": sequence_ids[seq_idx],
                "sequence": codon_seq,
                "activations": example_acts[key],
                "max_activation": max(example_acts[key]) if example_acts[key] else 0.0,
            }
        )

example_rows.sort(key=lambda r: (r["feature_id"], r["example_rank"]))
examples_table = pa.table(
    {
        "feature_id": pa.array([r["feature_id"] for r in example_rows], type=pa.int32()),
        "example_rank": pa.array([r["example_rank"] for r in example_rows], type=pa.int8()),
        "protein_id": pa.array([r["protein_id"] for r in example_rows]),
        "sequence": pa.array([r["sequence"] for r in example_rows]),
        "activations": pa.array([r["activations"] for r in example_rows], type=pa.list_(pa.float32())),
        "max_activation": pa.array([r["max_activation"] for r in example_rows], type=pa.float32()),
    }
)
pq.write_table(
    examples_table, output_dir / "feature_examples.parquet", row_group_size=N_EXAMPLES * 100, compression="snappy"
)

print(f"Wrote {len(meta_rows)} features, {len(example_rows)} examples")

## Enrich Atlas with Analysis Results

If you've already run notebooks 02 (codon analysis) and 03 (gene enrichment), their output files can be loaded and merged into the atlas. This adds columns like `gsea_overall_best`, `gsea_GO_Biological_Process`, etc., which become available as color/filter dimensions in the dashboard.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq


atlas_path = output_dir / "features_atlas.parquet"
table = pq.read_table(atlas_path)
n = table.num_rows
print(f"Atlas has {n} features, {len(table.column_names)} columns")

# Add GSEA columns if gene_enrichment_report.json exists
gsea_report_path = Path("../outputs/1b_layer16/gene_enrichment/gene_enrichment_report.json")
if gsea_report_path.exists():
    with open(gsea_report_path) as f:
        gsea_data = json.load(f)

    # Build label lookup: feature_idx -> best term name
    best_labels = {}
    for entry in gsea_data.get("per_feature", []):
        idx = entry["feature_idx"]
        ob = entry.get("overall_best")
        if ob and ob.get("term_name"):
            best_labels[idx] = ob["term_name"]

    gsea_col = [best_labels.get(i, "unlabeled") for i in range(n)]
    if "gsea_overall_best" in table.column_names:
        table = table.drop("gsea_overall_best")
    table = table.append_column("gsea_overall_best", pa.array(gsea_col))
    print(f"  Added GSEA labels: {sum(1 for l in gsea_col if l != 'unlabeled')} features labeled")

    pq.write_table(table, atlas_path, compression="snappy")
    print(f"  Updated {atlas_path}")
else:
    print(f"  No GSEA report found at {gsea_report_path}")
    print("  Run notebook 03_gene_enrichment.ipynb first to add GSEA labels.")

## Launch Dashboard

The dashboard is a React app served by a Python backend. Point it at the output directory containing the parquet files.

In [ ]:
print("Dashboard files generated:")
print(f"  {output_dir / 'features_atlas.parquet'}")
print(f"  {output_dir / 'feature_metadata.parquet'}")
print(f"  {output_dir / 'feature_examples.parquet'}")
print()
print("To launch the dashboard, run:")
print(f"  python scripts/launch_dashboard.py --data-dir {output_dir}")
print()
print("Or for a production run with variant analysis (uses the dashboard.py script):")
print("  python scripts/dashboard.py \\")
print("      --checkpoint outputs/1b_layer16/checkpoints/checkpoint_final.pt \\")
print("      --model-path $MODEL_PATH \\")
print("      --csv-path $CSV_PATH \\")
print("      --layer 16 --num-sequences 2000 \\")
print("      --output-dir outputs/1b_layer16/dashboard")

## Next Steps

- **05_auto_interp.ipynb** — Generate natural-language feature descriptions with an LLM
- Explore the dashboard interactively: color the UMAP by GSEA labels, click features to see their top sequences, filter by activation frequency